# BattleActs Actuarial Exam Preparation Products
### DATASCI 530 — Project 1
Maxine Hakimi-Bulson

This project compares six BattleActs exam preparation courses: Exams 5, 6C, 6U, 7, 8, and 9. It collects listed prices, access periods, practice-exam counts, advertised study tools, and public PDF and Excel sample contents.

Data were collected on October 19, 2026. BattleActs granted written permission for this educational scraping project and for sharing the code, dataset, and analysis on GitHub. Collection was limited to public product pages and all linked public PDF and Excel samples. Accounts, paid course materials, customer data, and video downloads were excluded. Requests were spaced with pauses.


## Data Collection Process

Before coding, I manually inspected two products and organized their information in a spreadsheet to choose the variables.

The script opens the Exams menu and uses a loop to click each of the six product links. It extracts course information and public sample-card metadata, then downloads every linked PDF and Excel file once per unique URL. Videos and cards without URLs are excluded from file downloads. Requests are spaced with pauses.

Three datasets are exported: Products has one row per course; PDF Samples and Excel Samples each have one row per unique file. Shared destinations retain all advertised labels in one record. Product URLs link samples to courses. PDF text and cell values from every Excel worksheet are retained; spreadsheet formulas are stored as expressions and are not executed. Extracted text may omit images, charts, text boxes, and visual layout.

Files are stored in `sample_files/PDF/[exam]/` and `sample_files/Excel/[exam]/` with descriptive names. Recognized downloads from earlier versions are moved into these folders and reused. No unrelated files are deleted.

Tool indicators record mentions in product descriptions, feature lists, and sample labels. “Not matched” does not establish that a feature is unavailable. The notebook requires pandas, selenium, requests, pypdf, and openpyxl. Only the five results tables are displayed.


In [ ]:
# Import packages
import pandas as pd
from IPython.display import HTML, display
import time
from selenium import webdriver
import re
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import requests
import hashlib
from pathlib import Path
from urllib.parse import urlsplit, unquote
from pypdf import PdfReader
from openpyxl import load_workbook

# Display only concise, formatted report tables
def show_table(frame):
    table = frame.to_html(index=False, na_rep="—", border=0, escape=True)
    style = """<style>
    .report-table table {border-collapse:collapse;width:100%;font-size:13px;}
    .report-table th {background:#eaf0f7;color:#183e60;text-align:left;}
    .report-table th,.report-table td {padding:9px;border-bottom:1px solid #dce3eb;white-space:normal;}
    .report-table tr:nth-child(even) {background:#f7f9fb;}
    </style>"""
    display(HTML(style + '<div class="report-table">' + table + '</div>'))

# Open Chrome and navigate to BattleActs
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=options)

starting_url = "https://battleactsmain.ca/v2/home"
driver.get(starting_url)

## Product Extraction Function

In [ ]:
def extract_product(driver):
    # Return None when an optional text field is absent
    def get_text(xpath):
        elements = driver.find_elements("xpath", xpath)
        return elements[0].text.strip() if elements else None

    # Basic product information
    product_name = get_text("//h1")
    price_text = get_text('//h2[contains(., "$")]')

    price = (
        float(price_text.split()[0].replace("$", "").replace(",", ""))
        if price_text else None
    )
    currency = price_text.split()[-1] if price_text else None

    # Description and its bullet points
    description_parts = driver.find_elements(
        "xpath",
        '//h2[contains(., "Study Guide")]/parent::div/p'
        ' | //h2[contains(., "Study Guide")]/parent::div//li'
    )
    description = "\n".join(
        element.text.strip() for element in description_parts
    ) or None

    # Overview feature bullets
    feature_items = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Overview Of Our Features"]'
        '/parent::div/parent::div//ul/li'
    )
    features = "\n".join(
        item.text.strip() for item in feature_items
    ) or None

    # Structured fields derived from the collected text
    feature_text = features or ""
    duration_match = re.search(
        r"(\d+)-month'?s?\s+access", feature_text, re.I
    )
    extension_match = re.search(
        r"free\s+(\d+)-month\s+extension", feature_text, re.I
    )
    practice_match = re.search(
        r"(\d+)\s+original practice exams?", feature_text, re.I
    )

    # Exam-specific preparation description
    exam_differences = get_text(
        '//p[contains(., "different than other")]'
        '/following-sibling::p[1]'
    )

    product_url = driver.current_url
    collection_date = pd.Timestamp.now(
        tz="America/New_York"
    ).isoformat(timespec="seconds")

    product = {
        "product_name": product_name,
        "price": price,
        "currency": currency,
        "description": description,
        "features": features,
        "duration_months": (
            int(duration_match.group(1)) if duration_match else None
        ),
        "product_url": product_url,
        "collection_date": collection_date,
        "exam_differences": exam_differences,
        "practice_exam_count": (
            int(practice_match.group(1)) if practice_match else None
        ),
        "free_extension_months": (
            int(extension_match.group(1)) if extension_match else None
        )
    }

    # Sample cards within the section containing the sample heading
    sample_cards = driver.find_elements(
        "xpath",
        '//h2[normalize-space()="Sample our Study Platform"]'
        '/ancestor::div[contains(concat(" ", normalize-space(@class), " "),'
        ' " product-section ")][1]'
        '//div[contains(concat(" ", normalize-space(@class), " "),'
        ' " owl-item ")'
        ' and not(contains(concat(" ", normalize-space(@class), " "),'
        ' " cloned "))]'
    )

    samples = []

    for card in sample_cards:
        labels = card.find_elements("xpath", ".//p")
        controls = card.find_elements(
            "xpath", ".//a[@href] | .//button[@data-url]"
        )

        if not labels or not controls:
            raise ValueError(
                "Unexpected sample-card structure on " + product_url
            )

        control = controls[0]
        sample_url = (
            control.get_attribute("href")
            or control.get_attribute("data-url")
        )
        url_path = (sample_url or "").split("?")[0].split("#")[0].lower()

        if url_path.endswith(".pdf"):
            resource_type = "PDF"
        elif url_path.endswith((".xlsx", ".xls")):
            resource_type = "Excel"
        elif url_path.endswith(".mp4"):
            resource_type = "Video"
        else:
            resource_type = "Unknown"

        samples.append({
            "product_name": product_name,
            "sample_label": labels[0].get_attribute("textContent").strip(),
            "button_text": control.get_attribute("textContent").strip(),
            "sample_url": sample_url,
            "resource_type": resource_type,
            "product_url": product_url,
            "collection_date": collection_date
        })

    return product, samples

## Automated Navigation and Collection

In [ ]:
wait = WebDriverWait(driver, 15)

# Open the homepage and collect the exam-link destinations
driver.get(starting_url)

menu_xpath = '//a[@id="navbarDropdown" and normalize-space()="Exams"]'
links_xpath = (
    '//a[contains(@class, "dropdown-item")'
    ' and contains(@href, "/v2/product/")]'
)

wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

exam_links = driver.find_elements("xpath", links_xpath)

# Remove repeated URLs while preserving their order
exam_urls = list(dict.fromkeys(
    link.get_attribute("href") for link in exam_links
))

if len(exam_urls) != 6:
    raise ValueError(
        f"Expected 6 exam links, but found {len(exam_urls)}. Check the menu."
    )

products = []
all_samples = []

for exam_url in exam_urls:
    # Reopen the homepage and menu for each product
    driver.get(starting_url)
    wait.until(EC.element_to_be_clickable((By.XPATH, menu_xpath))).click()

    # Navigate by actually clicking the product link
    product_link_xpath = (
        '//a[contains(@class, "dropdown-item")'
        f' and @href="{exam_url}"]'
    )
    wait.until(
        EC.element_to_be_clickable((By.XPATH, product_link_xpath))
    ).click()

    wait.until(EC.url_to_be(exam_url))
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h1")))
    time.sleep(2)  # Allow the sample carousel to initialize

    product, samples = extract_product(driver)

    # Preserve completed rows as the loop progresses
    products.append(product)
    all_samples.extend(samples)


    time.sleep(3)  # Pause between products

# Build the two datasets
products_df = pd.DataFrame(products)
samples_df = pd.DataFrame(all_samples)



## Prepare Study-Tool Categories

In [ ]:
has_url = samples_df["sample_url"].fillna("").str.strip().ne("")
coming_soon = samples_df["button_text"].str.contains("coming soon", case=False, na=False)
samples_df["sample_status"] = "No URL listed"
samples_df.loc[has_url, "sample_status"] = "Link listed"
samples_df.loc[~has_url & coming_soon, "sample_status"] = "Coming soon"
# Classify selected tools using product text and sample labels
sample_labels = samples_df.groupby("product_url")["sample_label"].agg("\n".join)
tool_text = (
    products_df[["description", "features"]].fillna("").agg("\n".join, axis=1)
    + "\n" + products_df["product_url"].map(sample_labels).fillna("")
)
products_final = products_df.copy()
for column, pattern in {
    "instructional_video_mentioned": r"video lessons|instructional videos?",
    "formula_summary_sheet_mentioned": r"formula|summary sheets?",
    "hybrid_exam_mentioned": r"hybrid CAS exam"
}.items():
    products_final[column] = tool_text.str.contains(pattern, case=False, na=False)


## Download and Extract PDF and Excel Samples

In [ ]:
# Select all linked PDF and Excel samples; exclude videos and missing URLs
selected_samples = samples_df.loc[
    samples_df["resource_type"].isin(["PDF", "Excel"])
    & samples_df["sample_url"].fillna("").str.strip().ne("")
].copy()

sample_folder = Path("sample_files")
sample_folder.mkdir(exist_ok=True)
content_records = []

for _, sample in selected_samples.drop_duplicates("sample_url").iterrows():
    url = sample["sample_url"]
    suffix = Path(unquote(urlsplit(url).path)).suffix.lower()
    exam_name = re.sub(r"[^A-Za-z0-9]+", "_", sample["product_name"]).strip("_")
    label = sample["sample_label"].lower()
    if "study manual" in label:
        sample_name = "Study_manual"
    elif "formula" in label or "summary sheet" in label:
        sample_name = "Formula_summary_sheet"
    elif "past cas exams" in label:
        sample_name = "Past_CAS_exams"
    else:
        sample_name = "Practice_problems_and_exams"
    format_folder = "PDF" if suffix == ".pdf" else "Excel"
    exam_folder = sample_folder / format_folder / exam_name
    exam_folder.mkdir(parents=True, exist_ok=True)
    local_path = exam_folder / f"{sample_name}{suffix}"

    # Reuse and organize files produced by previous notebook versions
    old_key = hashlib.sha256(url.encode()).hexdigest()[:12]
    previous_paths = [
        sample_folder / f"{exam_name}_{sample_name}{suffix}",
        sample_folder / f"{old_key}{suffix}"
    ]
    if not local_path.exists():
        for previous_path in previous_paths:
            if previous_path.exists():
                previous_path.rename(local_path)
                break
    record = {
        "sample_url": url,
        "content_collection_date": pd.Timestamp.now(tz="America/New_York").isoformat(timespec="seconds"),
        "extraction_status": "Failed",
        "page_count": None, "sheet_count": None,
        "sheet_names": None, "formula_cells": None, "extracted_text": None
    }

    try:
        if not local_path.exists():
            response = requests.get(
                url, headers={"User-Agent": "DATASCI530 educational project"}, timeout=60
            )
            response.raise_for_status()
            local_path.write_bytes(response.content)

        if suffix == ".pdf":
            reader = PdfReader(local_path)
            record["page_count"] = len(reader.pages)
            text = "\n".join(page.extract_text() or "" for page in reader.pages)

        elif suffix == ".xlsx":
            workbook = load_workbook(local_path, read_only=True, data_only=False)
            try:
                record["sheet_count"] = len(workbook.sheetnames)
                record["sheet_names"] = " | ".join(workbook.sheetnames)
                text_parts = []
                formula_count = 0
                for sheet in workbook.worksheets:
                    text_parts.append(f"Worksheet: {sheet.title}")
                    for row in sheet.iter_rows():
                        values = []
                        for cell in row:
                            if cell.value is not None:
                                values.append(str(cell.value))
                                formula_count += int(cell.data_type == "f")
                        if values:
                            text_parts.append(" | ".join(values))
                record["formula_cells"] = formula_count
                text = "\n".join(text_parts)
            finally:
                workbook.close()
        else:
            raise ValueError(f"Unsupported format: {suffix}")

        record["extracted_text"] = text.strip() or None
        record["extraction_status"] = "Extracted" if text.strip() else "No text extracted"

    except Exception:
        record["extraction_status"] = "Failed"

    content_records.append(record)
    time.sleep(3)

content_df = pd.DataFrame(content_records)


## Build and Check the Three Final Datasets

In [ ]:
# One record per unique file; retain all labels for shared destinations
file_metadata = selected_samples.groupby("sample_url", sort=False).agg(
    product_name=("product_name", "first"),
    product_url=("product_url", "first"),
    resource_type=("resource_type", "first"),
    collection_date=("collection_date", "first"),
    sample_label=("sample_label", lambda labels: " | ".join(dict.fromkeys(labels)))
).reset_index()
files_final = file_metadata.merge(
    content_df, on="sample_url", how="left", validate="one_to_one"
)

common_columns = [
    "product_name", "sample_label", "sample_url", "product_url",
    "collection_date", "content_collection_date", "extraction_status", "extracted_text"
]
pdf_samples = files_final.loc[
    files_final["resource_type"].eq("PDF"), common_columns + ["page_count"]
].copy()
excel_samples = files_final.loc[
    files_final["resource_type"].eq("Excel"),
    common_columns + ["sheet_count", "sheet_names", "formula_cells"]
].copy()

# Check distinct files, product links, and extraction coverage
assert products_final["product_url"].is_unique
assert len(products_final) == len(products_df)
assert len(pdf_samples) + len(excel_samples) == selected_samples["sample_url"].nunique()
for frame in [pdf_samples, excel_samples]:
    assert frame["sample_url"].is_unique
    assert frame["product_url"].isin(products_final["product_url"]).all()
assert set(files_final.loc[
    files_final["extraction_status"].eq("Extracted"), "sample_url"
]) == set(content_df.loc[
    content_df["extraction_status"].eq("Extracted"), "sample_url"
])


## Results

### Table 1. Exam Preparation Course Overview

The six courses below are the products collected in this project. Prices are listed in their original currencies. The extension is conditional; seven months of standard access should not be interpreted as thirteen months of guaranteed access.

In [ ]:
# Course-by-course overview: each row is one exam preparation product
course_overview = products_final[[
    "product_name", "price", "currency", "duration_months",
    "practice_exam_count", "free_extension_months"
]].rename(columns={
    "product_name": "Exam prep course",
    "price": "Listed price",
    "currency": "Currency",
    "duration_months": "Access (months)",
    "practice_exam_count": "Original practice exams",
    "free_extension_months": "Conditional extension (months)"
})
course_overview["Listed price"] = course_overview["Listed price"].map("{:,.2f}".format)
show_table(course_overview)


Exam prep course,Listed price,Currency,Access (months),Original practice exams,Conditional extension (months)
CAS Exam 5,500.00,USD,7,4,6
CAS Exam 6C,"1,000.00",CAD,7,1,6
CAS Exam 6U,"1,000.00",USD,7,2,6
CAS Exam 7,"1,000.00",USD,7,2,6
CAS Exam 8,"1,000.00",USD,7,2,6
CAS Exam 9,"1,000.00",USD,7,2,6


### Table 2. Dataset Completeness

PDF and Excel measurements are stored separately so that each dataset contains only relevant columns. A failed or empty extraction remains visible in the quality table.

In [ ]:
quality_records = []
for name, frame in {
    "Products": products_final, "PDF samples": pdf_samples, "Excel samples": excel_samples
}.items():
    blank_cells = frame.apply(
        lambda column: column.map(lambda value: isinstance(value, str) and not value.strip())
    )
    source_column = "product_url" if name == "Products" else "sample_url"
    quality_records.append({
        "Dataset": name, "Observations": len(frame),
        "Missing/blank cells": int((frame.isna() | blank_cells).sum().sum()),
        "Duplicate source URLs": int(frame[source_column].duplicated().sum()),
        "Failed extractions": int(frame["extraction_status"].ne("Extracted").sum())
            if name != "Products" else 0
    })
show_table(pd.DataFrame(quality_records))


Dataset,Observations,Missing/blank cells,Duplicate source URLs,Failed extractions
Products,6,0,0,0
PDF samples,9,0,0,0
Excel samples,6,0,0,0


## Data Quality

The collection contains six products and 25 advertised sample cards. Sixteen cards link to 15 unique non-video files: nine PDFs and six Excel workbooks. Two Exam 5 cards share one PDF; their labels are retained together rather than duplicating the file record. The eight video cards were excluded from downloads. One Exam 9 card is marked “Coming Soon” and has no URL. It is not counted as an available file.

All 15 files yielded text. PDF and Excel measurements are stored in separate datasets. Excel text includes every worksheet; worksheet counts include instructions and contents sheets, not just practice questions. Sample-file size and structure do not measure course quality or coverage.

Exam 5's price, access period, conditional extension, and practice-exam count were compared with the manual collection. Duplicate source URLs, missing values, dataset relationships, and extraction coverage are checked in the code. Currency is retained separately from numeric prices.


### Table 3. Numeric Summaries

Variances are sample variances. Price variances use squared currency units. The CAD variance is undefined with one observation.

In [ ]:
# Prices are kept separate by currency; other measures use all products
summary_records = []
for currency, group in products_final.groupby("currency"):
    values = group["price"]
    summary_records.append({"Measure": f"Price ({currency})",
                            "Observations": values.count(),
                            "Mean": values.mean(), "Sample variance": values.var()})
for column, label in {
    "duration_months": "Standard access (months)",
    "practice_exam_count": "Original practice exams",
    "free_extension_months": "Conditional extension (months)"
}.items():
    values = products_final[column]
    summary_records.append({"Measure": label, "Observations": values.count(),
                            "Mean": values.mean(), "Sample variance": values.var()})
show_table(pd.DataFrame(summary_records).round(2))


Measure,Observations,Mean,Sample variance
Price (CAD),1,1000.00,—
Price (USD),5,900.00,50000.00
Standard access (months),6,7.00,0.00
Original practice exams,6,2.17,0.97
Conditional extension (months),6,6.00,0.00


### Table 4. Selected Study Tools

These indicators capture mentions, not verified availability. Features common to all courses remain in the original description text rather than repeated indicator columns.

In [ ]:
tool_table = products_final.set_index("product_name")[[
    "instructional_video_mentioned", "formula_summary_sheet_mentioned", "hybrid_exam_mentioned"
]].rename(columns={
    "instructional_video_mentioned": "Instructional videos",
    "formula_summary_sheet_mentioned": "Formula/summary sheets",
    "hybrid_exam_mentioned": "Hybrid exam"
}).replace({True: "Mentioned", False: "Not matched"})
tool_table.index.name = "Exam prep course"
show_table(tool_table.reset_index())


Exam prep course,Instructional videos,Formula/summary sheets,Hybrid exam
CAS Exam 5,Mentioned,Mentioned,Not matched
CAS Exam 6C,Not matched,Not matched,Not matched
CAS Exam 6U,Mentioned,Mentioned,Not matched
CAS Exam 7,Not matched,Mentioned,Mentioned
CAS Exam 8,Not matched,Not matched,Mentioned
CAS Exam 9,Not matched,Not matched,Mentioned


### Table 5. Public PDF and Excel Samples by Course

Counts refer to unique destination files, not advertised cards. Page and worksheet ranges describe the available samples; they do not measure the size or quality of the paid course.

In [ ]:
# Compact overview by course; file-specific measurements stay in their own CSV
sample_rows = []
for _, product in products_final.iterrows():
    pdfs = pdf_samples.loc[pdf_samples["product_url"].eq(product["product_url"])]
    workbooks = excel_samples.loc[excel_samples["product_url"].eq(product["product_url"])]
    pages = pdfs["page_count"].dropna()
    sheets = workbooks["sheet_count"].dropna()
    sample_rows.append({
        "Exam prep course": product["product_name"],
        "PDF files": len(pdfs),
        "PDF pages (range)": f"{int(pages.min())}–{int(pages.max())}" if len(pages) else "—",
        "Excel files": len(workbooks),
        "Worksheets (range)": f"{int(sheets.min())}–{int(sheets.max())}" if len(sheets) else "—"
    })
show_table(pd.DataFrame(sample_rows))


Exam prep course,PDF files,PDF pages (range),Excel files,Worksheets (range)
CAS Exam 5,2,2–17,1,32–32
CAS Exam 6C,1,7–7,1,11–11
CAS Exam 6U,2,2–5,2,3–29
CAS Exam 7,2,2–9,1,4–4
CAS Exam 8,1,6–6,1,16–16
CAS Exam 9,1,7–7,0,—


## Discussion

Five courses are priced in USD, averaging $900; Exam 6C is listed separately at CAD 1,000. All six list seven months of standard access and a conditional six-month extension. Original practice-exam counts range from one to four.

The collected text shows differences in mentions of instructional videos, formula/summary sheets, and hybrid exams. The 25 public sample cards include 24 listed destinations and one Coming Soon card. Text was extracted from all 15 unique linked PDF and Excel files, covering 16 cards, for later content comparisons.

These data describe one provider at one point in time. Advertised features and selected excerpts do not measure course quality, exam difficulty, or student outcomes. Sample sizes and document structures are not directly comparable measures of course coverage.


## Export Final Datasets

The submission datasets are Products, PDF Samples, and Excel Samples. The original combined samples CSV is superseded.

In [ ]:
products_final.to_csv("battleacts_products.csv", index=False)
pdf_samples.to_csv("battleacts_pdf_samples.csv", index=False)
excel_samples.to_csv("battleacts_excel_samples.csv", index=False)


In [ ]:
driver.quit()